# 01.6 — Error handling

Reading Python tracebacks and using `try / except / else / finally`. This notebook is the single most useful one in Module 01 because **debugging a Python failure is mostly about reading the traceback**, and the format is different enough from MATLAB errors that it deserves dedicated practice.

**Prerequisite:** [01.1 syntax basics](01.1_syntax_basics.ipynb).

## Section 1 — What MATLAB does

MATLAB errors look like:

```
Error using my_func (line 17)
Undefined function or variable 'foo'.

Error in cgg_runAutoEncoder (line 312)
    result = my_func(foo);
```

Two-line format: the immediate error at the top, plus the calling line below. Caught with `try / catch`:

```matlab
try
    result = my_func(foo);
catch ME
    fprintf('Caught: %s\n', ME.message);
end
```

Python's equivalent is structurally similar but the traceback format is different — typically much longer because Python shows every stack frame, top-to-bottom.

## Section 2 — The Python concepts you need

### 2.1 — Reading a Python traceback

Python tracebacks read **top to bottom**: each frame is a function call going deeper. The actual error is at the **very bottom**. Reading bottom-up is fastest:

```
Traceback (most recent call last):
  File "/tmp/example.py", line 6, in <module>
    main()
  File "/tmp/example.py", line 4, in main
    return work(x)
  File "/tmp/example.py", line 2, in work
    return x / 0
ZeroDivisionError: division by zero          ← THE ACTUAL ERROR
```

Three things to extract:

1. **Error class** (`ZeroDivisionError`) — usually identifies the bug class immediately.
2. **Error message** (`division by zero`) — disambiguates within that class.
3. **The deepest frame** (`File "/tmp/example.py", line 2, in work`) — where the error physically happened.

Then read frames upward to understand the call chain that got there.

Trigger one yourself:

In [ ]:
def divide(a, b):
    return a / b

def main():
    return divide(10, 0)

main()    # → ZeroDivisionError

### 2.2 — `try / except` — catching errors

The keyword is `except`, not `catch`. The exception object is bound with `as`.

In [ ]:
try:
    result = 10 / 0
except ZeroDivisionError as e:
    print(f"Caught: {type(e).__name__} — {e}")

In [ ]:
# Multiple exception types
def safe_div(a, b):
    try:
        return a / b
    except ZeroDivisionError:
        return float("inf")
    except TypeError as e:
        print(f"Type problem: {e}")
        return None

print(safe_div(10, 2))
print(safe_div(10, 0))
print(safe_div(10, "x"))

**`except Exception:` catches everything** in Python's exception hierarchy. **Don't do this** unless you have a very specific reason — it hides bugs you'd rather see. Catch the specific exception you expect.

### 2.3 — `else` and `finally`

A `try` block can have all four parts:

```python
try:
    # the risky operation
except SomeError as e:
    # handle the specific error
else:
    # runs ONLY if no exception was raised
finally:
    # ALWAYS runs (clean up resources)
```

`else` is rare but useful when you want post-success logic that shouldn't be in the `try` (so exceptions in it don't get swallowed). `finally` is for cleanup — closing files, releasing locks. The `with` statement (covered in 01.8) usually replaces `finally`.

In [ ]:
def parse_int(s):
    try:
        n = int(s)
    except ValueError:
        print(f"  not a number: {s!r}")
        return None
    else:
        print(f"  parsed {s!r} → {n}")
        return n
    finally:
        print(f"  finished trying {s!r}")

parse_int("42")
print()
parse_int("nope")

### 2.4 — `raise` — throwing your own exceptions

Use `raise SomeError("message")` to signal a problem. Pick a specific exception class — `ValueError` for bad inputs, `KeyError` for missing dict keys, `RuntimeError` for "this shouldn't happen," etc.

In [ ]:
def set_fold(k: int):
    if k < 1:
        raise ValueError(f"fold must be >= 1 (MATLAB 1-indexed); got {k}.")
    return k

try:
    set_fold(0)
except ValueError as e:
    print(f"Caught: {e}")

**Re-raising:** sometimes you want to catch, log, and re-raise so the caller still sees the error. Use a bare `raise` inside the except block:

```python
try:
    risky()
except SomeError:
    log.error("risky failed; about to re-raise")
    raise         # propagates the original exception unchanged
```

**Wrapping with context:** `raise X("...") from original_exception` preserves the original cause and shows it in the traceback. The codebase uses this in `_translate_overrides` so a typo in a sweep entry produces a clear "this is the new error; caused by KeyError(...)" chain.

### 2.5 — Custom exceptions

For domain-specific errors, define your own exception class by inheriting from `Exception`:

In [ ]:
class SweepIndexOutOfRange(Exception):
    """Raised when a SLURM array task ID falls outside the sweep table."""

try:
    raise SweepIndexOutOfRange("sweep_index=999 (max=147)")
except SweepIndexOutOfRange as e:
    print(f"Caught domain error: {e}")

The codebase keeps custom exceptions to a minimum — `MatlabNotFoundError` in `interop/matlab_runner.py` is the main one. Most errors use built-in types (`ValueError`, `KeyError`, `IndexError`) because callers are more likely to catch those.

## Section 3 — The `neural_data_decoding` implementation

Look at the `_translate_overrides` function — it raises a `KeyError` with extra context when a sweep entry uses an unknown field name:

In [ ]:
from pathlib import Path
src = Path("../../src/neural_data_decoding/sweeps/dispatcher.py").read_text().splitlines()
# Find _translate_overrides
start = next((i for i, line in enumerate(src) if line.startswith("def _translate_overrides")), -1)
for i in range(start, min(start + 25, len(src))):
    print(f"{i + 1:3} | {src[i]}")

Notice:

* The `if matlab_key not in _MATLAB_TO_PYTHON_FIELD:` check is a guard.
* `raise KeyError(...)` is preferred over `assert ...` for production code — assertions can be stripped with `-O`, but a `raise` always fires.
* The error message names the offending key AND tells the developer how to fix it ("Add it to ... or fix the typo"). Good error messages save debugging time.

Now trigger the error to see the traceback:

In [ ]:
from neural_data_decoding.sweeps.dispatcher import _translate_overrides

try:
    _translate_overrides({"NotARealField": 42})
except KeyError as e:
    print(f"  type: {type(e).__name__}")
    print(f"  message: {e}")

## Section 4 — Hands-on exercises

### Exercise 4.1 — port a MATLAB try / catch

Port:

```matlab
try
    n = my_parse('abc');
    fprintf('parsed %d\n', n);
catch ME
    fprintf('failed: %s\n', ME.message);
end
```

In [ ]:
# Your attempt here


In [ ]:
# Reveal:
try:
    n = int("abc")
    print(f"parsed {n}")
except ValueError as e:
    print(f"failed: {e}")

### Exercise 4.2 — bottom-up traceback reading

Run the cell below. Identify (a) the error class, (b) the error message, and (c) the line where the error physically happened. Don't read the source first — practice extracting it from the traceback.

In [ ]:
def make_config():
    return {"hidden": [1000, 500, 250]}

def get_first_layer(cfg):
    return cfg["hiddne"][0]    # typo: "hiddne"

cfg = make_config()
get_first_layer(cfg)

Answer: the error is `KeyError: 'hiddne'` on the `cfg["hiddne"][0]` line. The traceback's deepest frame points at the typo.

### Exercise 4.3 — `raise` with a helpful message

Write a function `pick_fold(k, num_folds)` that returns `k` if it's in `[1, num_folds]` and raises `ValueError` with a useful message otherwise. The message should include both the bad value and the valid range.

In [ ]:
# Your attempt here


In [ ]:
# Reveal:
def pick_fold(k: int, num_folds: int) -> int:
    if not (1 <= k <= num_folds):
        raise ValueError(
            f"fold must be in [1, {num_folds}]; got {k}."
        )
    return k

try:
    pick_fold(15, 10)
except ValueError as e:
    print(e)

## Section 5 — Common errors

### "Exception caught, but I still see the traceback"

You either re-raised with a bare `raise`, or you printed the exception inside `except` without preventing it from propagating further. Make sure the `except` block doesn't `raise` again unless you intend to.

### `except Exception:` catching too much

You may have meant to catch only a specific failure. Catching `Exception` swallows `KeyboardInterrupt`'s less-aggressive cousin `Exception` (which includes user-visible bugs). Be specific.

### `except` without `as`

```python
except ValueError:    # OK if you don't need the exception object
    handle()

except ValueError as e:    # OK and lets you read e.args, str(e), etc.
    print(e)
```

Both are valid. Use the `as` form whenever you might want to log the message.

### Empty `except:` (no exception type)

```python
try:
    risky()
except:           # ← caught EVERYTHING including KeyboardInterrupt and SystemExit
    pass
```

This is a **bug**. It silently masks `Ctrl+C`. Always specify a type, even if it's just `except Exception:`.

### `raise` without anything

A bare `raise` outside of an `except` block is a SyntaxError (Python 3). Inside an `except`, bare `raise` re-raises the active exception.

## Section 6 — Further reading

- [Python tutorial — Errors and exceptions](https://docs.python.org/3/tutorial/errors.html). The official walkthrough.
- [Built-in exceptions](https://docs.python.org/3/library/exceptions.html). The exception class hierarchy — useful for picking the right one to raise.
- [PEP 3134 — Exception chaining](https://peps.python.org/pep-3134/). Explains `raise X from Y` and `__cause__`.

Next up: **[01.7 dataclasses and typed configs](01.7_dataclasses_and_typed_configs.ipynb)**.